# Region-Level Sufficiency in Distilled and Pruned Landslide Detection

Full experimental execution and analysis notebook implementing `experimental_design.md`:

- **T0**: SAM2-UNet (`sam2_hiera_l`, `352x352`) teacher training & offline logit export (`128x128`)
- **2x2 Factorial Design** on RGB Attention U-Net (`filters=32`):
  - **S1**: No-KD, Unpruned (`train.py`)
  - **S2**: DIST-KD, Unpruned (`train_kd.py`)
  - **S3**: No-KD, HRank Pruned (`rho=0.5`, `rho_center=0.25`) + Task-only FT
  - **S4**: DIST-KD, HRank Pruned (`rho=0.5`, `rho_center=0.25`) + KD FT (Primary candidate)
- **Additional Baselines & Ablations**:
  - **S4-TaskFT**: KD before pruning, task-only FT after pruning
  - **S5**: Capacity-matched scratch baseline using S4's exact `channel_cfg.json` (RQ4)
  - **A1**: Compression rate sweep on S2 (`rho in {0.30, 0.50, 0.70}`)
  - **A2**: Binary DIST inter-class ablation (`kd_beta = 0.0` vs. `1.0`)
- **Multi-seed protocol** (`SEEDS = [42, 123, 2026]`), standardized test evaluation, mask export, statistical tests (Wilcoxon & exact McNemar), Tables 1–3, and diagnostic Figures E.2, E.3, E.4, and E.6.

## 0. Colab Environment & Dataset Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/thesis
!git clone https://github.com/pratamabintang/L4SKD.git
%cd /content/drive/MyDrive/thesis/L4SKD

In [ ]:
# !git fetch origin
# !git reset --hard origin/main

In [ ]:
!pip install -q -r requirements.txt -r requirements-teacher.txt

In [ ]:
!mkdir -p /content/datasets/landslide4senses

!rsync -a --quiet \
"/content/drive/MyDrive/thesis/datasets/landslide4senses/" \
"/content/datasets/landslide4senses/" 

## 1. Experiment Configuration & Resumable Runner Helper

Pre-experiment fixed constants (`experimental_design.md` Section 3) and a resumable command helper that skips completed stages unless `FORCE_RERUN = True`.

In [ ]:
import csv
import json
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Pre-experiment fixed constants (experimental_design.md Section 3)
SEEDS = [42, 123, 2026]
TEACHER_DIR = "runs/sam2unet_l_rgb"
COMPRESS_RATE = 0.5
CENTER_RATE = 0.25
MIN_CHANNELS = 2
HRANK_BATCHES = 10
IOU_TAU = 0.5
DILATION_ALPHA = 0.1

# Supplementary ablations
A1_RATES = [0.30, 0.70]  # 0.50 is reused directly from S4

# Export prediction panels + uint8 HDF5 masks for qualitative analysis (seed 42 by default; set to SEEDS for all)
EXPORT_MASK_SEEDS = [42]

# Set True to overwrite existing completed runs
FORCE_RERUN = False


def run_cmd(cmd: list[str], skip_if_exists: str | Path | list[str | Path] | None = None) -> None:
    """Run a CLI command unless all target artifacts in `skip_if_exists` already exist."""
    if skip_if_exists is not None and not FORCE_RERUN:
        targets = (
            [Path(p) for p in skip_if_exists]
            if isinstance(skip_if_exists, (list, tuple))
            else [Path(skip_if_exists)]
        )
        if targets and all(t.exists() for t in targets):
            print(f"[SKIP] Already exists: {', '.join(str(t) for t in targets)}")
            return
    print("[RUN]", " ".join(cmd))
    subprocess.run(cmd, check=True)


def s1_name(seed: int) -> str:
    return f"s1_nokd_f32_seed{seed}"


def s2_name(seed: int) -> str:
    return f"s2_kd_f32_seed{seed}"


def pruned_s1_dir(seed: int) -> str:
    return f"runs/pruned_s1_seed{seed}"


def pruned_s2_dir(seed: int) -> str:
    return f"runs/pruned_s2_seed{seed}"


def s3_name(seed: int) -> str:
    return f"s3_nokd_pruned_seed{seed}"


def s4_name(seed: int) -> str:
    return f"s4_kd_pruned_seed{seed}"


def s4_taskft_name(seed: int) -> str:
    return f"s4_taskft_seed{seed}"


def s5_name(seed: int) -> str:
    return f"s5_scratch_small_seed{seed}"


def a1_pruned_dir(seed: int, rho: float) -> str:
    tag = int(round(rho * 100))
    return f"runs/pruned_s2_rho{tag}_seed{seed}"


def a1_name(seed: int, rho: float) -> str:
    tag = int(round(rho * 100))
    return f"a1_s4_rho{tag}_seed{seed}"


def a2_s2_name(seed: int) -> str:
    return f"a2_s2_beta0_seed{seed}"


def a2_pruned_s2_dir(seed: int) -> str:
    return f"runs/pruned_a2_s2_beta0_seed{seed}"


def a2_s4_name(seed: int) -> str:
    return f"a2_s4_beta0_seed{seed}"

## 2. Step 0 — Teacher Training & Logit Export (`T0`, Once)

Downloads the pretrained `sam2_hiera_large.pt` checkpoint if missing, trains `SAM2-UNet` (`sam2_hiera_l.yaml`, `352x352`), evaluates on test, and exports clean `128x128` teacher logits (`teacher_logits_{train,val,test}.h5`).

In [ ]:
!mkdir -p checkpoints
!if [ ! -f checkpoints/sam2_hiera_large.pt ]; then wget -O checkpoints/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt; else echo "Checkpoint already exists."; fi

In [ ]:
run_cmd(
    [
        sys.executable,
        "train_teacher.py",
        "--config",
        "configs/train_teacher.yaml",
        "--seed",
        "42",
    ],
    skip_if_exists=[
        f"{TEACHER_DIR}/teacher_logits_train.h5",
        f"{TEACHER_DIR}/teacher_logits_val.h5",
        f"{TEACHER_DIR}/test_metrics.json",
    ],
)

## 3. Step 1 — Unpruned Student Arms (`S1` and `S2`) Across Seeds

- **S1**: No-KD, Unpruned (`filters=32`, task-only Dice loss)
- **S2**: DIST-KD, Unpruned (`filters=32`, offline DIST from `T0`)

In [ ]:
for seed in SEEDS:
    # S1: No-KD, Unpruned (filters=32)
    r1 = s1_name(seed)
    run_cmd(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/train_rgb.yaml",
            "--filters",
            "32",
            "--seed",
            str(seed),
            "--run-name",
            r1,
        ],
        skip_if_exists=f"runs/{r1}/test_metrics.json",
    )

    # S2: DIST-KD, Unpruned (filters=32)
    r2 = s2_name(seed)
    run_cmd(
        [
            sys.executable,
            "train_kd.py",
            "--config",
            "configs/train_kd_f32.yaml",
            "--teacher-dir",
            TEACHER_DIR,
            "--seed",
            str(seed),
            "--run-name",
            r2,
        ],
        skip_if_exists=f"runs/{r2}/test_metrics.json",
    )

## 4. Step 2 — HRank Structured Pruning (`S1 -> S3_init`, `S2 -> S4_init`) Across Seeds

Prunes each unpruned student with `compress_rate = 0.5`, `center_rate = 0.25`, `min_channels = 2`, `hrank_batches = 10`.

In [ ]:
for seed in SEEDS:
    # Prune No-KD student S1
    p1 = pruned_s1_dir(seed)
    run_cmd(
        [
            sys.executable,
            "prune_student.py",
            "--run-dir",
            f"runs/{s1_name(seed)}",
            "--compress-rate",
            str(COMPRESS_RATE),
            "--center-rate",
            str(CENTER_RATE),
            "--min-channels",
            str(MIN_CHANNELS),
            "--hrank-batches",
            str(HRANK_BATCHES),
            "--eval-val",
            "--output-dir",
            p1,
        ],
        skip_if_exists=[f"{p1}/pruned_init.weights.h5", f"{p1}/channel_cfg.json"],
    )

    # Prune KD student S2
    p2 = pruned_s2_dir(seed)
    run_cmd(
        [
            sys.executable,
            "prune_student.py",
            "--run-dir",
            f"runs/{s2_name(seed)}",
            "--compress-rate",
            str(COMPRESS_RATE),
            "--center-rate",
            str(CENTER_RATE),
            "--min-channels",
            str(MIN_CHANNELS),
            "--hrank-batches",
            str(HRANK_BATCHES),
            "--eval-val",
            "--output-dir",
            p2,
        ],
        skip_if_exists=[f"{p2}/pruned_init.weights.h5", f"{p2}/channel_cfg.json"],
    )

## 5. Step 3 — Recovery Fine-Tuning & Scratch-Small Baseline (`S3`, `S4`, `S4-TaskFT`, `S5`) Across Seeds

- **S3**: Pruned `S1` + Task-only fine-tuning (`lr=2e-4`, `epochs=100`, `patience=20`)
- **S4**: Pruned `S2` + DIST-KD fine-tuning (`finetune_pruned_kd.yaml`)
- **S4-TaskFT**: Pruned `S2` + Task-only fine-tuning (ablation isolating KD role during recovery)
- **S5**: Trained from scratch using `S4`'s exact `channel_cfg.json` (`RQ4` capacity-matched baseline)

In [ ]:
for seed in SEEDS:
    p1 = pruned_s1_dir(seed)
    p2 = pruned_s2_dir(seed)

    # S3: No-KD, Pruned + Task-only FT
    r3 = s3_name(seed)
    run_cmd(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/train_rgb.yaml",
            "--filters",
            "32",
            "--lr",
            "2e-4",
            "--epochs",
            "100",
            "--patience",
            "20",
            "--pretrained-weights",
            f"{p1}/pruned_init.weights.h5",
            "--channel-cfg",
            f"{p1}/channel_cfg.json",
            "--seed",
            str(seed),
            "--run-name",
            r3,
        ],
        skip_if_exists=f"runs/{r3}/test_metrics.json",
    )

    # S4: DIST-KD, Pruned + KD FT (primary candidate)
    r4 = s4_name(seed)
    run_cmd(
        [
            sys.executable,
            "train_kd.py",
            "--config",
            "configs/finetune_pruned_kd.yaml",
            "--teacher-dir",
            TEACHER_DIR,
            "--pretrained-weights",
            f"{p2}/pruned_init.weights.h5",
            "--channel-cfg",
            f"{p2}/channel_cfg.json",
            "--seed",
            str(seed),
            "--run-name",
            r4,
        ],
        skip_if_exists=f"runs/{r4}/test_metrics.json",
    )

    # S4-TaskFT: Ablation — KD before pruning, task-only FT after pruning
    r4_tft = s4_taskft_name(seed)
    run_cmd(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/train_rgb.yaml",
            "--filters",
            "32",
            "--lr",
            "2e-4",
            "--epochs",
            "100",
            "--patience",
            "20",
            "--pretrained-weights",
            f"{p2}/pruned_init.weights.h5",
            "--channel-cfg",
            f"{p2}/channel_cfg.json",
            "--seed",
            str(seed),
            "--run-name",
            r4_tft,
        ],
        skip_if_exists=f"runs/{r4_tft}/test_metrics.json",
    )

    # S5: RQ4 Baseline — Trained from scratch at identical pruned channel_cfg as S4
    r5 = s5_name(seed)
    run_cmd(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/train_rgb.yaml",
            "--filters",
            "32",
            "--channel-cfg",
            f"{p2}/channel_cfg.json",
            "--seed",
            str(seed),
            "--run-name",
            r5,
        ],
        skip_if_exists=f"runs/{r5}/test_metrics.json",
    )

## 6. Supplementary Ablations (`A1` Compression Rate Sweep & `A2` Binary DIST `kd_beta = 0.0`)

- **A1**: Prune `S2` at `rho in {0.30, 0.70}` (`0.50` is `S4`) with `center_rate = min(rho, 0.25)` followed by KD fine-tuning.
- **A2**: Repeat `S2` and `S4` with `kd_beta = 0.0` (intra-class term only) to test binary DIST inter-class collapse.

In [ ]:
# A1 — Compression Rate Sweep on S2 (rho in {0.30, 0.70}; 0.50 is S4)
for seed in SEEDS:
    for rho in A1_RATES:
        c_rate = min(rho, 0.25)
        p_dir = a1_pruned_dir(seed, rho)
        run_cmd(
            [
                sys.executable,
                "prune_student.py",
                "--run-dir",
                f"runs/{s2_name(seed)}",
                "--compress-rate",
                str(rho),
                "--center-rate",
                str(c_rate),
                "--min-channels",
                str(MIN_CHANNELS),
                "--hrank-batches",
                str(HRANK_BATCHES),
                "--eval-val",
                "--output-dir",
                p_dir,
            ],
            skip_if_exists=[f"{p_dir}/pruned_init.weights.h5", f"{p_dir}/channel_cfg.json"],
        )

        r_a1 = a1_name(seed, rho)
        run_cmd(
            [
                sys.executable,
                "train_kd.py",
                "--config",
                "configs/finetune_pruned_kd.yaml",
                "--teacher-dir",
                TEACHER_DIR,
                "--pretrained-weights",
                f"{p_dir}/pruned_init.weights.h5",
                "--channel-cfg",
                f"{p_dir}/channel_cfg.json",
                "--seed",
                str(seed),
                "--run-name",
                r_a1,
            ],
            skip_if_exists=f"runs/{r_a1}/test_metrics.json",
        )

In [ ]:
# A2 — Binary DIST Inter-Class Ablation (kd_beta = 0.0)
for seed in SEEDS:
    r_a2_s2 = a2_s2_name(seed)
    run_cmd(
        [
            sys.executable,
            "train_kd.py",
            "--config",
            "configs/train_kd_f32.yaml",
            "--teacher-dir",
            TEACHER_DIR,
            "--kd-beta",
            "0.0",
            "--seed",
            str(seed),
            "--run-name",
            r_a2_s2,
        ],
        skip_if_exists=f"runs/{r_a2_s2}/test_metrics.json",
    )

    p_a2 = a2_pruned_s2_dir(seed)
    run_cmd(
        [
            sys.executable,
            "prune_student.py",
            "--run-dir",
            f"runs/{r_a2_s2}",
            "--compress-rate",
            str(COMPRESS_RATE),
            "--center-rate",
            str(CENTER_RATE),
            "--min-channels",
            str(MIN_CHANNELS),
            "--hrank-batches",
            str(HRANK_BATCHES),
            "--eval-val",
            "--output-dir",
            p_a2,
        ],
        skip_if_exists=[f"{p_a2}/pruned_init.weights.h5", f"{p_a2}/channel_cfg.json"],
    )

    r_a2_s4 = a2_s4_name(seed)
    run_cmd(
        [
            sys.executable,
            "train_kd.py",
            "--config",
            "configs/finetune_pruned_kd.yaml",
            "--teacher-dir",
            TEACHER_DIR,
            "--kd-beta",
            "0.0",
            "--pretrained-weights",
            f"{p_a2}/pruned_init.weights.h5",
            "--channel-cfg",
            f"{p_a2}/channel_cfg.json",
            "--seed",
            str(seed),
            "--run-name",
            r_a2_s4,
        ],
        skip_if_exists=f"runs/{r_a2_s4}/test_metrics.json",
    )

## 7. Step 4 — Standardized Test Evaluation & Prediction Mask Export

Re-evaluates every student run on the `test` split with fixed constants (`--iou-tau 0.5 --dilation-alpha 0.1`), producing `metrics_test.json` and `per_patch_test.csv`, and exports full test-set visual panels and uint8 HDF5 masks (`--save-masks`) for qualitative inspection.

In [ ]:
all_student_runs = []
for seed in SEEDS:
    all_student_runs.extend(
        [
            s1_name(seed),
            s2_name(seed),
            s3_name(seed),
            s4_name(seed),
            s4_taskft_name(seed),
            s5_name(seed),
            a2_s2_name(seed),
            a2_s4_name(seed),
        ]
        + [a1_name(seed, rho) for rho in A1_RATES]
    )

for run_name_str in all_student_runs:
    run_cmd(
        [
            sys.executable,
            "test.py",
            "--run-dir",
            f"runs/{run_name_str}",
            "--split",
            "test",
            "--iou-tau",
            str(IOU_TAU),
            "--dilation-alpha",
            str(DILATION_ALPHA),
        ],
        skip_if_exists=[
            f"runs/{run_name_str}/metrics_test.json",
            f"runs/{run_name_str}/per_patch_test.csv",
        ],
    )

In [ ]:
core_arm_fns = [s1_name, s2_name, s3_name, s4_name, s4_taskft_name, s5_name]
for seed in EXPORT_MASK_SEEDS:
    for fn in core_arm_fns:
        r_name = fn(seed)
        run_cmd(
            [
                sys.executable,
                "predict.py",
                "--run-dir",
                f"runs/{r_name}",
                "--split",
                "test",
                "--num-samples",
                "-1",
                "--save-masks",
            ],
            skip_if_exists=f"runs/{r_name}/predictions_test/masks",
        )

## 8. Analysis, Statistical Tests, Results Tables (Tables 1–3) & Figures (`E.2`, `E.3`, `E.4`, `E.6`)

Aggregates results across `SEEDS` into `runs/analysis/`:
- **Table 1**: Main quantitative comparison across `T0`, `S1`–`S5`, `S4-TaskFT`, `A1`, and `A2`
- **RQ1–RQ4 Contrasts & Paired Hypothesis Tests**: Wilcoxon signed-rank & exact McNemar tests on positive test patches
- **Table 2**: Size-stratified region sufficiency (`small`, `medium`, `large`)
- **Table 3**: RQ5 metric-suite rank correlation matrix (Spearman $\rho$ and Kendall $\tau$)
- **Figures**: `fig_e2_interaction.png`, `fig_e3_pareto.png`, `fig_e4_hrank_profile.png`, `fig_e6_rq5_rankings.png`

In [ ]:
# 8.1 — Load Metrics & Build Table 1 (Main Quantitative Comparison + Ablations)
ANALYSIS_DIR = Path("runs/analysis")
FIGURES_DIR = ANALYSIS_DIR / "figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def load_run_metrics(run_dir: str | Path) -> dict | None:
    d = Path(run_dir)
    for fname in ("metrics_test.json", "test_metrics.json"):
        p = d / fname
        if p.is_file():
            return json.loads(p.read_text(encoding="utf-8"))
    return None


def extract_flat_metrics(m: dict) -> dict[str, float]:
    rs = m.get("region_sufficiency", {})
    so = m.get("standard_overlap", {})
    bc = m.get("boundary_comparators", {})
    mc = m.get("model_cost", {})
    params = mc.get("params", mc.get("total_params", float("nan")))
    flops = mc.get("flops", float("nan"))
    size_mb = mc.get("weights_size_mb", float("nan"))
    return {
        "params": float(params) if params is not None else float("nan"),
        "flops": float(flops) if flops is not None else float("nan"),
        "size_mb": float(size_mb) if size_mb is not None else float("nan"),
        "tol_iou": float(rs.get("tolerant_iou_rate", float("nan"))),
        "centroid_hit": float(rs.get("centroid_hit_rate", float("nan"))),
        "dilated_recall": float(rs.get("size_scaled_dilated_recall", float("nan"))),
        "empty_fp_rate": float(rs.get("empty_patch_fp_rate", float("nan"))),
        "std_iou": float(so.get("iou", float("nan"))),
        "dice_f1": float(so.get("dice_f1", float("nan"))),
        "boundary_iou": float(bc.get("boundary_iou_positive_mean", float("nan"))),
        "s_measure": float(bc.get("s_measure_all_mean", float("nan"))),
    }


ARMS_SPEC = [
    ("T0", [TEACHER_DIR]),
    ("S1", [f"runs/{s1_name(s)}" for s in SEEDS]),
    ("S2", [f"runs/{s2_name(s)}" for s in SEEDS]),
    ("S3", [f"runs/{s3_name(s)}" for s in SEEDS]),
    ("S4", [f"runs/{s4_name(s)}" for s in SEEDS]),
    ("S4-TaskFT", [f"runs/{s4_taskft_name(s)}" for s in SEEDS]),
    ("S5", [f"runs/{s5_name(s)}" for s in SEEDS]),
    ("A1-rho0.30", [f"runs/{a1_name(s, 0.30)}" for s in SEEDS]),
    ("A1-rho0.70", [f"runs/{a1_name(s, 0.70)}" for s in SEEDS]),
    ("A2-S2-beta0", [f"runs/{a2_s2_name(s)}" for s in SEEDS]),
    ("A2-S4-beta0", [f"runs/{a2_s4_name(s)}" for s in SEEDS]),
]

arm_records: dict[str, list[dict[str, float]]] = {}
arm_raw_metrics: dict[str, list[dict]] = {}
for arm_label, dirs in ARMS_SPEC:
    recs, raws = [], []
    for d in dirs:
        m = load_run_metrics(d)
        if m is not None:
            recs.append(extract_flat_metrics(m))
            raws.append(m)
    arm_records[arm_label] = recs
    arm_raw_metrics[arm_label] = raws


def fmt_mean_sd(vals: list[float], scale: float = 1.0, decimals: int = 4) -> str:
    clean = [v * scale for v in vals if not math.isnan(v)]
    if not clean:
        return "N/A"
    mean = float(np.mean(clean))
    if len(clean) == 1:
        return f"{mean:.{decimals}f}"
    sd = float(np.std(clean, ddof=1))
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f}"


metric_cols = [
    ("params", 1.0, 0),
    ("flops", 1e-6, 1),
    ("size_mb", 1.0, 2),
    ("tol_iou", 1.0, 4),
    ("centroid_hit", 1.0, 4),
    ("dilated_recall", 1.0, 4),
    ("empty_fp_rate", 1.0, 4),
    ("std_iou", 1.0, 4),
    ("dice_f1", 1.0, 4),
    ("boundary_iou", 1.0, 4),
    ("s_measure", 1.0, 4),
]

table1_rows = []
print(
    f"{'Arm':13s} | {'N':>2s} | {'Params':>11s} | {'MFLOPs':>9s} | {'MB':>6s} | "
    f"{'Tol.IoU':>15s} | {'Cent.Hit':>15s} | {'Dil.Rec':>15s} | "
    f"{'EmptyFP':>15s} | {'Std.IoU':>15s} | {'Dice/F1':>15s}"
)
print("-" * 155)
for arm_label, _ in ARMS_SPEC:
    recs = arm_records[arm_label]
    row = {"arm": arm_label, "n_seeds": len(recs)}
    for k, scale, dec in metric_cols:
        vals = [r[k] for r in recs]
        clean = [v * scale for v in vals if not math.isnan(v)]
        row[f"{k}_mean"] = float(np.mean(clean)) if clean else None
        row[f"{k}_sd"] = float(np.std(clean, ddof=1)) if len(clean) > 1 else None
        row[f"{k}_fmt"] = fmt_mean_sd(vals, scale=scale, decimals=dec)
    table1_rows.append(row)
    print(
        f"{arm_label:13s} | {len(recs):2d} | {row['params_fmt']:>11s} | {row['flops_fmt']:>9s} | "
        f"{row['size_mb_fmt']:>6s} | {row['tol_iou_fmt']:>15s} | {row['centroid_hit_fmt']:>15s} | "
        f"{row['dilated_recall_fmt']:>15s} | {row['empty_fp_rate_fmt']:>15s} | "
        f"{row['std_iou_fmt']:>15s} | {row['dice_f1_fmt']:>15s}"
    )

t1_fields = ["arm", "n_seeds"] + [
    f"{k}_{suffix}" for k, _, _ in metric_cols for suffix in ("mean", "sd", "fmt")
]
with (ANALYSIS_DIR / "table1_main.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=t1_fields)
    w.writeheader()
    w.writerows(table1_rows)
print(f"Saved {ANALYSIS_DIR / 'table1_main.csv'}")

In [ ]:
# 8.2 — RQ1–RQ4 Contrasts & Paired Statistical Tests (Wilcoxon & Exact McNemar)
def arm_mean(arm: str, key: str) -> float:
    vals = [r[key] for r in arm_records.get(arm, []) if not math.isnan(r[key])]
    return float(np.mean(vals)) if vals else float("nan")


contrast_keys = ["tol_iou", "centroid_hit", "dilated_recall", "std_iou", "dice_f1"]
print("=== Seed-Averaged Experimental Contrasts (RQ1–RQ4) ===")
contrasts = [
    ("RQ1: S2 - S1 (Pre-prune KD)", lambda k: arm_mean("S2", k) - arm_mean("S1", k)),
    ("RQ2a: S3 - S1 (Prune w/o KD)", lambda k: arm_mean("S3", k) - arm_mean("S1", k)),
    ("RQ2b: S4 - S2 (Prune w/ KD)", lambda k: arm_mean("S4", k) - arm_mean("S2", k)),
    (
        "RQ3: (S4-S3)-(S2-S1) (Interaction)",
        lambda k: (arm_mean("S4", k) - arm_mean("S3", k)) - (arm_mean("S2", k) - arm_mean("S1", k)),
    ),
    ("RQ3b: S4 - S4-TaskFT (KD Recovery)", lambda k: arm_mean("S4", k) - arm_mean("S4-TaskFT", k)),
    ("RQ4: S4 - S5 (KD vs Scratch-Small)", lambda k: arm_mean("S4", k) - arm_mean("S5", k)),
    ("Ref: S3 - S5 (Prune No-KD vs Scratch)", lambda k: arm_mean("S3", k) - arm_mean("S5", k)),
]
for label, fn in contrasts:
    vals_str = ", ".join(f"{k}={fn(k):+.4f}" for k in contrast_keys)
    print(f"  {label:38s} -> {vals_str}")


def load_positive_per_patch(run_dir: str | Path) -> dict[str, dict]:
    p = Path(run_dir) / "per_patch_test.csv"
    if not p.is_file():
        return {}
    out = {}
    with p.open("r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["is_positive"].strip().lower() == "true":
                out[row["sample_id"]] = {
                    "sample_iou": float(row["sample_iou"]),
                    "dilated_recall": float(row["dilated_recall"]),
                    "tolerant_iou": row["tolerant_iou"].strip().lower() == "true",
                    "centroid_hit": row["centroid_hit"].strip().lower() == "true",
                }
    return out


def paired_tests_across_seeds(arm_a_fn, arm_b_fn, label: str) -> dict:
    a_dil, b_dil = [], []
    a_iou, b_iou = [], []
    a_tol, b_tol = [], []
    a_cen, b_cen = [], []

    for seed in SEEDS:
        pa = load_positive_per_patch(f"runs/{arm_a_fn(seed)}")
        pb = load_positive_per_patch(f"runs/{arm_b_fn(seed)}")
        common = sorted(set(pa.keys()) & set(pb.keys()))
        for sid in common:
            a_dil.append(pa[sid]["dilated_recall"])
            b_dil.append(pb[sid]["dilated_recall"])
            a_iou.append(pa[sid]["sample_iou"])
            b_iou.append(pb[sid]["sample_iou"])
            a_tol.append(pa[sid]["tolerant_iou"])
            b_tol.append(pb[sid]["tolerant_iou"])
            a_cen.append(pa[sid]["centroid_hit"])
            b_cen.append(pb[sid]["centroid_hit"])

    if not a_dil:
        return {"comparison": label, "n_pairs": 0}

    def safe_wilcoxon(x, y):
        diff = np.asarray(x) - np.asarray(y)
        if np.allclose(diff, 0.0):
            return 0.0, 1.0
        res = stats.wilcoxon(x, y)
        return float(res.statistic), float(res.pvalue)

    def exact_mcnemar(x_bool, y_bool):
        xb = np.asarray(x_bool, dtype=bool)
        yb = np.asarray(y_bool, dtype=bool)
        n10 = int(np.sum(xb & ~yb))
        n01 = int(np.sum(~xb & yb))
        n_disc = n10 + n01
        if n_disc == 0:
            return n10, n01, 1.0
        p = float(stats.binomtest(min(n10, n01), n_disc, 0.5).pvalue)
        return n10, n01, p

    w_dil_stat, w_dil_p = safe_wilcoxon(a_dil, b_dil)
    w_iou_stat, w_iou_p = safe_wilcoxon(a_iou, b_iou)
    tol_10, tol_01, tol_p = exact_mcnemar(a_tol, b_tol)
    cen_10, cen_01, cen_p = exact_mcnemar(a_cen, b_cen)

    return {
        "comparison": label,
        "n_pairs": len(a_dil),
        "wilcoxon_dilated_recall_p": w_dil_p,
        "wilcoxon_sample_iou_p": w_iou_p,
        "mcnemar_tol_iou_n10": tol_10,
        "mcnemar_tol_iou_n01": tol_01,
        "mcnemar_tol_iou_p": tol_p,
        "mcnemar_centroid_n10": cen_10,
        "mcnemar_centroid_n01": cen_01,
        "mcnemar_centroid_p": cen_p,
    }


paired_specs = [
    ("S2 vs S1 (RQ1)", s2_name, s1_name),
    ("S3 vs S1 (RQ2a)", s3_name, s1_name),
    ("S4 vs S2 (RQ2b)", s4_name, s2_name),
    ("S4 vs S3 (KD+Prune vs NoKD+Prune)", s4_name, s3_name),
    ("S4 vs S4-TaskFT (RQ3 recovery)", s4_name, s4_taskft_name),
    ("S4 vs S5 (RQ4)", s4_name, s5_name),
    ("S3 vs S5 (RQ4 ref)", s3_name, s5_name),
]

stat_rows = [paired_tests_across_seeds(fa, fb, lbl) for lbl, fa, fb in paired_specs]
print("\n=== Paired Patch-Level Statistical Tests (Aggregated Positive Test Patches) ===")
for row in stat_rows:
    if row["n_pairs"] == 0:
        print(f"  {row['comparison']}: no per_patch_test.csv found")
        continue
    print(
        f"  {row['comparison']:36s} (n={row['n_pairs']}): "
        f"Wilcoxon(dil_rec p={row['wilcoxon_dilated_recall_p']:.4g}, iou p={row['wilcoxon_sample_iou_p']:.4g}) | "
        f"McNemar(tol_iou {row['mcnemar_tol_iou_n10']}:{row['mcnemar_tol_iou_n01']} p={row['mcnemar_tol_iou_p']:.4g}, "
        f"centroid {row['mcnemar_centroid_n10']}:{row['mcnemar_centroid_n01']} p={row['mcnemar_centroid_p']:.4g})"
    )

if stat_rows:
    with (ANALYSIS_DIR / "paired_stats.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(stat_rows[-1].keys()))
        w.writeheader()
        for r in stat_rows:
            if r["n_pairs"] > 0:
                w.writerow(r)

In [ ]:
# 8.3 — Table 2 (Size-Stratified Sufficiency) & Table 3 (RQ5 Rank Correlation Matrix)
STRATA = ["small", "medium", "large"]
STRATA_ARMS = ["T0", "S1", "S2", "S3", "S4", "S4-TaskFT", "S5"]

t2_rows = []
print("=== Table 2: Size-Stratified Region Sufficiency (Test Positive Patches) ===")
for stratum in STRATA:
    for arm in STRATA_ARMS:
        raws = arm_raw_metrics.get(arm, [])
        if not raws:
            continue
        n_patches = raws[0]["size_stratified"][stratum]["num_patches"]
        tol_vals = [m["size_stratified"][stratum]["tolerant_iou_rate"] for m in raws]
        cen_vals = [m["size_stratified"][stratum]["centroid_hit_rate"] for m in raws]
        dil_vals = [m["size_stratified"][stratum]["size_scaled_dilated_recall"] for m in raws]
        iou_vals = [m["size_stratified"][stratum]["mean_sample_iou"] for m in raws]
        row = {
            "stratum": stratum,
            "num_patches": n_patches,
            "arm": arm,
            "tol_iou_fmt": fmt_mean_sd([float(v) for v in tol_vals if v is not None]),
            "centroid_hit_fmt": fmt_mean_sd([float(v) for v in cen_vals if v is not None]),
            "dilated_recall_fmt": fmt_mean_sd([float(v) for v in dil_vals if v is not None]),
            "mean_sample_iou_fmt": fmt_mean_sd([float(v) for v in iou_vals if v is not None]),
        }
        t2_rows.append(row)
        print(
            f"  {stratum:6s} (n={n_patches:3d}) | {arm:9s} | "
            f"Tol={row['tol_iou_fmt']:15s} | Cent={row['centroid_hit_fmt']:15s} | "
            f"DilRec={row['dilated_recall_fmt']:15s} | SampleIoU={row['mean_sample_iou_fmt']:15s}"
        )

if t2_rows:
    with (ANALYSIS_DIR / "table2_size_stratified.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(t2_rows[0].keys()))
        w.writeheader()
        w.writerows(t2_rows)

# Table 3: RQ5 Metric-Suite Rank Correlation Matrix across {T0, S1, S2, S3, S4, S4-TaskFT, S5}
RQ5_ARMS = [a for a in ["T0", "S1", "S2", "S3", "S4", "S4-TaskFT", "S5"] if arm_records.get(a)]
RQ5_METRICS = [
    ("tol_iou", "Tolerant IoU"),
    ("centroid_hit", "Centroid Hit"),
    ("dilated_recall", "Dilated Recall"),
    ("std_iou", "Std. IoU"),
    ("dice_f1", "Dice / F1"),
    ("boundary_iou", "Boundary IoU"),
    ("s_measure", "S-measure"),
]

if len(RQ5_ARMS) >= 3:
    vecs = {
        key: np.array([arm_mean(arm, key) for arm in RQ5_ARMS], dtype=float)
        for key, _ in RQ5_METRICS
    }
    spearman_rows, kendall_rows = [], []
    print("\n=== Table 3: RQ5 Rank Correlations (Spearman rho / Kendall tau) ===")
    header = "Metric         | " + " | ".join(f"{lbl[:10]:>10s}" for _, lbl in RQ5_METRICS)
    print(header)
    print("-" * len(header))
    for k1, l1 in RQ5_METRICS:
        s_row, k_row = {"metric": l1}, {"metric": l1}
        cells_str = []
        for k2, l2 in RQ5_METRICS:
            rho_val = float(stats.spearmanr(vecs[k1], vecs[k2]).statistic)
            tau_val = float(stats.kendalltau(vecs[k1], vecs[k2]).statistic)
            s_row[l2] = round(rho_val, 4)
            k_row[l2] = round(tau_val, 4)
            cells_str.append(f"{rho_val:4.2f}/{tau_val:4.2f}")
        spearman_rows.append(s_row)
        kendall_rows.append(k_row)
        print(f"{l1:14s} | " + " | ".join(f"{c:>10s}" for c in cells_str))

    fieldnames_t3 = ["metric"] + [lbl for _, lbl in RQ5_METRICS]
    with (ANALYSIS_DIR / "table3_spearman.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames_t3)
        w.writeheader()
        w.writerows(spearman_rows)
    with (ANALYSIS_DIR / "table3_kendall.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames_t3)
        w.writeheader()
        w.writerows(kendall_rows)

In [ ]:
# 8.4 — Diagnostic Figures (E.2 Interaction, E.3 Pareto Frontier, E.4 HRank Profile, E.6 RQ5 Bump Chart)
def mean_sd(arm: str, key: str) -> tuple[float, float]:
    vals = [r[key] for r in arm_records.get(arm, []) if not math.isnan(r[key])]
    if not vals:
        return float("nan"), 0.0
    sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return float(np.mean(vals)), sd


# Figure E.2 — KD x Pruning Interaction Plot (RQ3)
if all(arm_records.get(a) for a in ("S1", "S2", "S3", "S4")):
    plot_metrics = [
        ("tol_iou", "Tolerant IoU (tau=0.5)"),
        ("centroid_hit", "Centroid Hit Rate"),
        ("dilated_recall", "Dilated Recall (alpha=0.1)"),
        ("std_iou", "Standard IoU"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    x = np.array([0, 1])
    for ax, (k, title) in zip(axes, plot_metrics):
        nokd_m = [mean_sd("S1", k)[0], mean_sd("S3", k)[0]]
        nokd_s = [mean_sd("S1", k)[1], mean_sd("S3", k)[1]]
        kd_m = [mean_sd("S2", k)[0], mean_sd("S4", k)[0]]
        kd_s = [mean_sd("S2", k)[1], mean_sd("S4", k)[1]]
        ax.errorbar(x, nokd_m, yerr=nokd_s, marker="o", capsize=4, label="No-KD (S1->S3)")
        ax.errorbar(x, kd_m, yerr=kd_s, marker="s", capsize=4, label="DIST-KD (S2->S4)")
        ax.set_xticks(x)
        ax.set_xticklabels(["Unpruned", "Pruned+FT"])
        ax.set_title(title)
        ax.grid(alpha=0.3)
    axes[0].legend(loc="best", fontsize=8)
    fig.suptitle("Figure E.2: KD x HRank Pruning Interaction (RQ3)", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_e2_interaction.png", dpi=150)
    plt.close(fig)

# Figure E.3 — Efficiency-Sufficiency Pareto Frontier
pareto_arms = [
    a
    for a in ("S1", "S2", "S3", "S4", "S4-TaskFT", "S5", "A1-rho0.30", "A1-rho0.70")
    if arm_records.get(a)
]
if pareto_arms:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    y_keys = [
        ("tol_iou", "Tolerant IoU (tau=0.5)"),
        ("centroid_hit", "Centroid Hit Rate"),
        ("dilated_recall", "Dilated Recall (alpha=0.1)"),
    ]
    a1_curve = [a for a in ("S2", "A1-rho0.30", "S4", "A1-rho0.70") if a in pareto_arms]
    for ax, (yk, ytitle) in zip(axes, y_keys):
        if len(a1_curve) >= 2:
            cx = [mean_sd(a, "flops")[0] * 1e-6 for a in a1_curve]
            cy = [mean_sd(a, yk)[0] for a in a1_curve]
            ax.plot(cx, cy, "--", color="gray", alpha=0.7, label="S2->A1 Frontier")
        for arm in pareto_arms:
            fx = mean_sd(arm, "flops")[0] * 1e-6
            fy, fsd = mean_sd(arm, yk)
            ax.errorbar(fx, fy, yerr=fsd, marker="o", capsize=3)
            ax.annotate(arm, (fx, fy), textcoords="offset points", xytext=(4, 4), fontsize=8)
        ax.set_xlabel("MFLOPs")
        ax.set_ylabel(ytitle)
        ax.grid(alpha=0.3)
    axes[0].legend(loc="best", fontsize=8)
    fig.suptitle("Figure E.3: Efficiency-Sufficiency Pareto Frontier", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_e3_pareto.png", dpi=150)
    plt.close(fig)

# Figure E.4 — Layer-Wise HRank Profiles & Width Reduction (Seed 42 S2)
hrank_npz = Path(pruned_s2_dir(SEEDS[0])) / "hrank_scores.npz"
prune_rep = Path(pruned_s2_dir(SEEDS[0])) / "pruning_report.json"
if hrank_npz.is_file() and prune_rep.is_file():
    rep = json.loads(prune_rep.read_text(encoding="utf-8"))
    npz = np.load(hrank_npz)
    units = list(rep["orig_channel_cfg"].keys())
    mean_ranks = [float(np.mean(npz[f"rank_{u}"])) for u in units]
    orig_ch = [int(rep["orig_channel_cfg"][u]) for u in units]
    pruned_ch = [int(rep["pruned_channel_cfg"][u]) for u in units]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    idx = np.arange(len(units))
    ax1.bar(idx, mean_ranks, color="steelblue")
    ax1.set_ylabel("Mean Feature Rank")
    ax1.set_title(f"Figure E.4: Layer-Wise HRank Profiles & Surviving Channels ({pruned_s2_dir(SEEDS[0])})")
    ax1.grid(axis="y", alpha=0.3)

    w = 0.38
    ax2.bar(idx - w / 2, orig_ch, width=w, label="Original")
    ax2.bar(idx + w / 2, pruned_ch, width=w, label="Pruned")
    ax2.set_ylabel("Channels")
    ax2.set_xticks(idx)
    ax2.set_xticklabels(units, rotation=45, ha="right")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_e4_hrank_profile.png", dpi=150)
    plt.close(fig)

# Figure E.6 — RQ5 Metric Ranking Comparison (Bump Chart)
if len(RQ5_ARMS) >= 3:
    fig, ax = plt.subplots(figsize=(11, 5))
    x_pos = np.arange(len(RQ5_METRICS))
    ranks_by_metric = {}
    for k, _ in RQ5_METRICS:
        vals = np.array([arm_mean(arm, k) for arm in RQ5_ARMS], dtype=float)
        # Rank 1 = highest score
        ranks_by_metric[k] = len(RQ5_ARMS) + 1 - stats.rankdata(vals, method="average")

    for i, arm in enumerate(RQ5_ARMS):
        y_ranks = [ranks_by_metric[k][i] for k, _ in RQ5_METRICS]
        ax.plot(x_pos, y_ranks, marker="o", linewidth=2, label=arm)
    ax.invert_yaxis()
    ax.set_yticks(range(1, len(RQ5_ARMS) + 1))
    ax.set_xticks(x_pos)
    ax.set_xticklabels([lbl for _, lbl in RQ5_METRICS], rotation=20, ha="right")
    ax.set_ylabel("Model Rank (1 = Best)")
    ax.set_title("Figure E.6: RQ5 Metric Ranking Comparison Across Arms")
    ax.grid(alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_e6_rq5_rankings.png", dpi=150)
    plt.close(fig)

print("Generated figures in:", sorted(p.name for p in FIGURES_DIR.glob("*.png")))